### 두 클래스를 연결해 전체 파이프라인이 실제 관광 API JSON에서도 정상 작동하는지 검증하는 통합 테스트


In [ ]:
from pathlib import Path
import json

import pandas as pd

from src.tour_data_etl import TourDataETL
from src.data_profiler import DataProfiler

# 출력 파일 경로
input_path = (
    Path("data")
    / "tour_api_sample.json"
)

parquet_output_path = (
    Path("outputs")
    / "tour_api_integrated.parquet"
)

metadata_output_path = (
    Path("outputs")
    / "tour_api_integrated_metadata.json"
)


print("입력 파일:", input_path)
print("입력 파일 존재 여부:", input_path.exists())

print(
    "Parquet 출력 경로:",
    parquet_output_path,
)

print(
    "메타데이터 출력 경로:",
    metadata_output_path,
)

입력 파일: data\tour_api_sample.json
입력 파일 존재 여부: True
Parquet 출력 경로: outputs\tour_api_integrated.parquet
메타데이터 출력 경로: outputs\tour_api_integrated_metadata.json


In [ ]:
etl = TourDataETL() #로드·변환·Parquet 저장 담당
profiler = DataProfiler() #프로파일링·태깅·메타데이터 저장 담당


# 1. 실제 관광 API JSON 로드
raw_df = etl.load_data(
    input_path
)

print("로드 결과 크기:", raw_df.shape)


# 2. 중첩 구조 평탄화 및 리스트 행 분리
clean_df = etl.transform(
    raw_df
)

print("변환 결과 크기:", clean_df.shape)

print("\n변환된 컬럼:")
print(clean_df.columns.tolist())


# 3. dict 또는 list가 남아 있는지 확인
nested_value_exists = clean_df.map(
    lambda value: isinstance(
        value,
        (dict, list),
    )
).any().any()

print(
    "\n중첩 값 존재 여부:",
    bool(nested_value_exists),
)

로드 결과 크기: (1, 1)
변환 결과 크기: (5, 30)

변환된 컬럼:
['response.header.resultCode', 'response.header.resultMsg', 'response.body.numOfRows', 'response.body.pageNo', 'response.body.totalCount', 'response.body.items.item.addr1', 'response.body.items.item.addr2', 'response.body.items.item.areacode', 'response.body.items.item.cat1', 'response.body.items.item.cat2', 'response.body.items.item.cat3', 'response.body.items.item.contentid', 'response.body.items.item.contenttypeid', 'response.body.items.item.createdtime', 'response.body.items.item.firstimage', 'response.body.items.item.firstimage2', 'response.body.items.item.cpyrhtDivCd', 'response.body.items.item.mapx', 'response.body.items.item.mapy', 'response.body.items.item.mlevel', 'response.body.items.item.modifiedtime', 'response.body.items.item.sigungucode', 'response.body.items.item.tel', 'response.body.items.item.title', 'response.body.items.item.zipcode', 'response.body.items.item.lDongRegnCd', 'response.body.items.item.lDongSignguCd', 'respons

In [ ]:
# 주요 컬럼 확인
display(
    clean_df[
        [
            "response.body.items.item.title", # 관광지 이름
            "response.body.items.item.addr1", # 주소
            "response.body.items.item.mapx", # 경도
            "response.body.items.item.mapy", # 위도
        ]
    ]
)

,response.body.items.item.title,response.body.items.item.addr1,response.body.items.item.mapx,response.body.items.item.mapy
0,가마오름,제주특별자치도 제주시 한경면 청수리,126.2466098987,33.3057279944
1,가문이오름(감은이오름),제주특별자치도 서귀포시 표선면 가시리,126.6876111209,33.4018299117
2,가새기오름,제주특별자치도 제주시 오라삼동,126.5038611983,33.4893700755
3,가세오름,제주특별자치도 서귀포시 표선면 녹산로 554,126.7231832338,33.3931758821
4,가시리국산화풍력발전단지,제주특별자치도 서귀포시 표선면 녹산로 464-78,126.7341100468,33.3939370848


In [7]:
# parquet 파일로 저장과 프로파일링, 태그 생성
# 1. 변환 데이터 Parquet 저장
saved_parquet_path = etl.save_to_parquet(
    clean_df,
    parquet_output_path,
)

print(
    "Parquet 저장 완료:",
    saved_parquet_path,
)


# 2. 변환된 데이터 프로파일링
stats = profiler.profile_data(
    clean_df
)

tags = profiler.generate_tags(
    stats
)

print("\n프로파일 결과:")
print(
    json.dumps(
        stats,
        ensure_ascii=False,
        indent=2,
    )
)

print("\n생성된 태그:")
print(tags)

Parquet 저장 완료: outputs\tour_api_integrated.parquet


100%|██████████| 30/30 [00:00<00:00, 160.91it/s]



프로파일 결과:
{
  "row_count": 5,
  "column_count": 30,
  "column_names": [
    "response.header.resultCode",
    "response.header.resultMsg",
    "response.body.numOfRows",
    "response.body.pageNo",
    "response.body.totalCount",
    "response.body.items.item.addr1",
    "response.body.items.item.addr2",
    "response.body.items.item.areacode",
    "response.body.items.item.cat1",
    "response.body.items.item.cat2",
    "response.body.items.item.cat3",
    "response.body.items.item.contentid",
    "response.body.items.item.contenttypeid",
    "response.body.items.item.createdtime",
    "response.body.items.item.firstimage",
    "response.body.items.item.firstimage2",
    "response.body.items.item.cpyrhtDivCd",
    "response.body.items.item.mapx",
    "response.body.items.item.mapy",
    "response.body.items.item.mlevel",
    "response.body.items.item.modifiedtime",
    "response.body.items.item.sigungucode",
    "response.body.items.item.tel",
    "response.body.items.item.title",
   

In [8]:
# 메타데이터 JSON 저장
dataset_info = {
    "id": "TOUR_API_01",
    "name": "한국관광공사 제주 관광지 API 샘플",
    "source_file": input_path.name,
}


saved_metadata_path = (
    profiler.export_metadata(
        dataset_info=dataset_info,
        profile_stats=stats,
        tags=tags,
        out_meta_path=metadata_output_path,
    )
)

print(
    "메타데이터 저장 완료:",
    saved_metadata_path,
)

메타데이터 저장 완료: outputs\tour_api_integrated_metadata.json


In [9]:
# 컬럼 확인
print("수치형 컬럼:")
print(stats["numeric_columns"])

print(
    "수치형 컬럼 수:",
    stats["numeric_column_count"],
)

수치형 컬럼:
['response.body.numOfRows', 'response.body.pageNo', 'response.body.totalCount']
수치형 컬럼 수: 3


In [10]:
# 전체 결과 검증
# Parquet 재로드
parquet_check = pd.read_parquet(
    saved_parquet_path
)


# 메타데이터 JSON 재로드
with saved_metadata_path.open(
    "r",
    encoding="utf-8",
) as file:
    metadata_check = json.load(file)


# Parquet 검증
assert parquet_check.shape == clean_df.shape

assert (
    parquet_check.columns.tolist()
    == clean_df.columns.tolist()
)

assert len(parquet_check) == 5


# 중첩 객체가 남았는지 확인
nested_value_exists = parquet_check.map(
    lambda value: isinstance(
        value,
        (dict, list),
    )
).any().any()

assert not bool(nested_value_exists)


# 메타데이터 검증
assert set(
    metadata_check.keys()
) == {
    "dataset_info",
    "profile_stats",
    "tags",
}

assert (
    metadata_check["dataset_info"]
    == dataset_info
)

assert (
    metadata_check["profile_stats"]
    == stats
)

assert metadata_check["tags"] == tags


print("Parquet 재로드 결과:", parquet_check.shape)
print("메타데이터 태그:", metadata_check["tags"])
print("전체 ETL 및 프로파일링 파이프라인 검증 통과")

Parquet 재로드 결과: (5, 30)
메타데이터 태그: ['multivariate']
전체 ETL 및 프로파일링 파이프라인 검증 통과
